
# AGN unification: the torus screens the disc with inclination

The composable AGN runner sums disc + broad/narrow lines + FeII + torus,
but a real dusty torus also *obscures the central engine* along edge-on
sightlines while its own infrared emission is not re-extinguished by that
same screen. ``tengri`` applies this inclination-dependent **torus screen**
automatically whenever the torus is one of the two CIGALE production grids
(``skirtor`` or ``fritz``); it closes the "disc + torus composed additively,
no torus screen on disc" gap (#294).

Here a single Stalevski+2016 (``skirtor``) torus reprocesses a fixed
Kubota & Done (2018) disc at ``log L_bol = 12.5`` (in log L_sun). Only the
viewing angle ``agn_cos_inc`` changes, sweeping from face-on (Type 1,
``cos i = 1``) to edge-on (Type 2, ``cos i -> 0``):

- **Type 1 (face-on):** the UV/optical disc + broad lines are seen directly.
- **Type 2 (edge-on):** the torus rim screens the central engine, so the
  UV/optical collapses by several dex behind an SMC reddening curve, while
  the mid-IR torus bump is essentially preserved.

Geometry (``skirtor2016`` convention): the sightline enters the torus when
``cos i < sin(oa)``, with ``oa`` the half-opening angle from the equator.
The Type-1/Type-2 transition is a sigmoid in ``cos i`` (C^1, gradient-safe
for HMC/VI), so a default face-on model is left unchanged.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

C_AA_PER_S = 2.998e18

# Inclination sweep: cos i = 1 (face-on, Type 1) -> 0.05 (edge-on, Type 2).
COS_INC = [1.0, 0.7, 0.5, 0.3, 0.05]
COLORS = plt.cm.RdYlBu(np.linspace(0.92, 0.05, len(COS_INC)))

SFH = {"type": "const", "all_params": tengri.FIXED, "log_total_mass": -10.0}
DUST = {"type": "two_component", "all_params": tengri.FIXED, "tau_diff": 0.0, "tau_bc": 0.0}

ssp = tengri.load_ssp()
fig, ax = plt.subplots(figsize=(7.2, 4.6))

# The Type-1/Type-2 boundary is cos(i) = sin(oa); read the torus's default
# opening angle so the label tracks the actual model rather than a magic number.
cos_inc_boundary = None

for cos_inc, color in zip(COS_INC, COLORS):
    model = tengri.SEDModel.build(
        ssp,
        sfh=SFH,
        dust=DUST,
        agn={
            "disc": {"type": "multicolor", "all_params": tengri.FIXED},
            "torus": {"type": "skirtor", "all_params": tengri.FIXED},
            "all_params": tengri.FIXED,
            "log_lbol": 12.5,
            "lum_ratio": 1.0,
            "cos_inc": cos_inc,
        },
        redshift=tengri.Fixed(0.05),
    )
    p = dict(model.spec.sample(jax.random.PRNGKey(0)))
    if cos_inc_boundary is None:
        cos_inc_boundary = float(np.sin(np.radians(float(p["agn_oa_skirtor"]))))
    out = model.predict(p)
    wave = np.asarray(model.wavelengths)
    nu_l_nu = C_AA_PER_S / wave * np.asarray(out.rest_sed())
    incl_deg = np.degrees(np.arccos(cos_inc))
    kind = "Type 1" if cos_inc > cos_inc_boundary else "Type 2"
    ax.loglog(wave, nu_l_nu, color=color, lw=1.5, label=rf"$i={incl_deg:.0f}^\circ$ ({kind})")

# Band markers: where the screening bites (UV/optical) vs where it does not (MIR).
for aa, name in [(3.0e3, "UV"), (5.5e3, "V"), (1.0e5, r"10 $\mu$m")]:
    ax.axvline(aa, color="0.85", lw=0.4, alpha=0.6)
    ax.text(aa, 4e46, name, fontsize=7, color="0.5", ha="center", va="top")

ax.set(
    xlim=(1e3, 3e6),
    ylim=(1e40, 5e46),
    xlabel=r"Rest-frame wavelength $\lambda$ [$\mathrm{\AA}$]",
    ylabel=r"$\nu L_\nu$  [erg s$^{-1}$]",
    title="Torus screen on the central engine (SKIRTOR, fixed $L_{\\rm bol}$)",
)
ax.legend(frameon=False, fontsize=8, loc="lower center", ncol=2)

fig.tight_layout()
plt.savefig("plot_torus_screen_disc.png", dpi=150, bbox_inches="tight")